# Publication snapshot v0.4.0
整理自既有研究；已移除outputs、憑證與私人路徑。見README與docs/limitations.md。
保留歷史實驗設計供檢視，不代表已修正資料切分或已在乾淨環境重跑。


# 蒸餾 Phase 3 — 學生模型推論 / 效果驗收

跑完 `02_train_qlora.ipynb` 之後用這本檢查訓好的學生模型：
1. 在 Phase 2 切出的 historical validation split (not an independent test) 上跑準確率 / confusion matrix。
2. 看幾筆答錯的例子，直覺感受哪裡學得不好。
3. 提供 `judge_student(source, summary)`，可以直接複製貼到「實驗1」notebook 的 detectors 裡，在 FaithBench 金標上跟 GPT-4o / qwen3:4b 一起比。

**這本一樣要在你的 GPU 電腦上跑**（讀 4bit 模型 + adapter）。


In [ ]:
import sys
print(sys.executable)


In [ ]:
import os, json, time, torch

# 跟 02_train_qlora.ipynb 保持一致，才能重建同一個 test split
STUDENT_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
DATA_JSONL    = "distill_data.jsonl"
ADAPTER_DIR   = "student_halluc_detector"   # Phase 2 存 adapter 的地方
MAX_SEQ_LEN   = 2048
SEED          = 42

SYS = "You are a strict factual-consistency checker. Given a SOURCE article and a SUMMARY, answer with exactly one word: CONSISTENT or HALLUCINATED."
def user_msg(source, summary):
    return f"SOURCE:\n{source}\n\nSUMMARY:\n{summary}\n\nIs the summary factually consistent with the source? Answer CONSISTENT or HALLUCINATED."
LABEL_TEXT = {0: "CONSISTENT", 1: "HALLUCINATED"}

print("device:", "cuda" if torch.cuda.is_available() else "⚠️ 沒偵測到 CUDA")


## 1. 重建 Phase 2 用的 test split（沿用舊驗證切分；非獨立測試）

In [ ]:
from datasets import Dataset

rows = []
with open(DATA_JSONL, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r.get("teacher_label") in (0, 1):
            rows.append({"source": r["source"], "summary": r["summary"],
                         "label": int(r["teacher_label"])})

ds_all = Dataset.from_list(rows)
split = ds_all.train_test_split(test_size=0.15, seed=SEED)
test_set = split["test"]
print("test set 筆數：", len(test_set), "｜ 0/1 分佈：",
      sum(x==0 for x in test_set["label"]), sum(x==1 for x in test_set["label"]))


## 2. 載入 base model + LoRA adapter

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, quantization_config=bnb,
                                            device_map="auto", torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print("loaded on", model.device)


## 3. 推論函式（也是要接回「實驗1」notebook 的那個）

In [ ]:
@torch.no_grad()
def judge_student(source, summary, max_new_tokens=5):
    """回傳 0=CONSISTENT, 1=HALLUCINATED，附推論秒數。給實驗1 notebook 的 detectors 用。"""
    msgs = [{"role": "system", "content": SYS},
            {"role": "user", "content": user_msg(source, summary)}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.pad_token_id)
    dt = time.time() - t0
    gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).upper()
    pred = 1 if "HALLUCINAT" in gen else 0
    return pred, dt, gen.strip()

# sanity check 一筆
p, dt, raw = judge_student(test_set[0]["source"], test_set[0]["summary"])
print("pred:", p, "| 秒數:", round(dt,2), "| 原始輸出:", raw, "| 真實標籤:", test_set[0]["label"])


## 4. 在 test set 上跑一輪，算準確率 / confusion matrix

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

preds, latencies, raws = [], [], []
for i in range(len(test_set)):
    ex = test_set[i]
    p, dt, raw = judge_student(ex["source"], ex["summary"])
    preds.append(p); latencies.append(dt); raws.append(raw)
    if (i+1) % 20 == 0:
        print(f"{i+1}/{len(test_set)}")

labels = test_set["label"]
print("\nAccuracy:", accuracy_score(labels, preds))
print("\nConfusion matrix (rows=真實, cols=預測), [0=CONSISTENT,1=HALLUCINATED]:")
print(confusion_matrix(labels, preds))
print("\n", classification_report(labels, preds, target_names=["CONSISTENT","HALLUCINATED"]))
print("平均每筆推論秒數:", sum(latencies)/len(latencies))


## 5. 看幾筆答錯的例子（直覺判斷哪裡學不好）

In [ ]:
wrong_idx = [i for i in range(len(test_set)) if preds[i] != labels[i]]
print(f"答錯 {len(wrong_idx)}/{len(test_set)} 筆")

for i in wrong_idx[:5]:
    ex = test_set[i]
    print("="*60)
    print("真實標籤:", LABEL_TEXT[ex["label"]], "| 模型預測:", LABEL_TEXT[preds[i]], "| 原始輸出:", raws[i])
    print("summary:", ex["summary"][:300])


## 6. 下一步

- 這裡的準確率是「跟 GPT-4o 老師標籤的一致度」（蒸餾保真度），還不是外部效度。
- 把上面 `judge_student` 複製貼到 `00_faithbench_baselines.ipynb` 的 detectors 區塊，在 **FaithBench 金標**上跑一次，才能跟 `naive_numcheck` / `ollama qwen3:4b` / `yours_gpt4o_detect` 放在同一張表比較（這才是論文/面試要講的「對金標準度」）。
- `latencies` 這組數字就是你「準度 vs 成本」故事要用的延遲數據，記得跟 GPT-4o API 的估計成本放在一起比。
